In [0]:
df = spark.read.json("/databricks-datasets/structured-streaming/events")
display(df)

In [0]:
df.createOrReplaceTempView('pokemon')

In [0]:
query = '''
select ingestion_date, poke.*
from pokemon
lateral view explode(results) as poke
qualify row_number() over (partition by poke.name order by ingestion_date desc) = 1
'''

df_new = spark.sql(query).coalesce(1)

df_new.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('bronze.pokemon.pokemon')

In [0]:
%sql
create database bronze.pokemon